In [1]:
from manim import *
import os

class LSTMDynamicFlow(MovingCameraScene):
    def construct(self):
        # ==========================================
        # 1. STYLES & CONFIG
        # ==========================================
        self.camera.background_color = "#111111"
        
        # Colors
        C_GATE    = "#FCE883"   # Yellow (Sigmoid)
        C_TANH    = "#EDB2E0"   # Pink (Tanh)
        C_HL      = "#FFFF00"   # Bright Yellow for active flow
        C_LINE    = WHITE
        C_TEXT    = GRAY_B
        
        # Layout Constants
        CODE_SCALE = 0.5
        DIAGRAM_SCALE = 0.70
        # Shifted further right
        DIAGRAM_SHIFT = RIGHT * 3.8
        
        # ==========================================
        # 2. HELPER FUNCTIONS
        # ==========================================
        def create_gate_box(label, sublabel, color, position):
            box = RoundedRectangle(corner_radius=0.15, width=1.0, height=0.8, 
                                 fill_color=color, fill_opacity=1, stroke_color=WHITE, stroke_width=2)
            box.move_to(position)
            # INCREASED FONT SIZE for box labels
            lbl = MathTex(label, color=BLACK, font_size=36).move_to(box.get_top() + DOWN*0.2)
            sub = MathTex(sublabel, color=BLACK, font_size=30).move_to(box.get_bottom() + UP*0.2)
            return VGroup(box, lbl, sub)

        def create_op(symbol, position):
            circle = Circle(radius=0.25, color=WHITE, fill_color=BLACK, fill_opacity=1, stroke_width=2)
            circle.move_to(position)
            # INCREASED FONT SIZE for op symbols
            sym = MathTex(symbol, color=WHITE, font_size=40).move_to(position)
            return VGroup(circle, sym)
            
        def elbow_path(start, end, color=WHITE, direction="horizontal_first"):
            if direction == "horizontal_first":
                mid = [end[0], start[1], 0]
            else:
                mid = [start[0], end[1], 0]
            p1 = Line(start, mid, color=color, stroke_width=2)
            p2 = Line(mid, end, color=color, stroke_width=2)
            joint = Dot(mid, radius=0.04, color=color)
            return VGroup(p1, p2, joint)

        def animate_flow(mob_path, color=C_HL, run_time=0.8):
            path_to_trace = VMobject()
            if isinstance(mob_path, VGroup):
                lines = [m for m in mob_path if isinstance(m, Line)]
                if len(lines) >= 2:
                    pts = [lines[0].get_start(), lines[0].get_end(), lines[-1].get_end()]
                    path_to_trace.set_points_as_corners(pts)
                elif len(lines) == 1:
                     path_to_trace = lines[0].copy()
                else: return
            else:
                path_to_trace = mob_path.copy()

            path_to_trace.set_stroke(color=color, width=5, opacity=1)
            self.play(ShowPassingFlash(path_to_trace, time_width=0.4), run_time=run_time)

        # INCREASED default font_size to 32
        def add_label(text, mob, direction=UP, buff=0.1, font_size=32, color=C_TEXT):
            l = MathTex(text, font_size=font_size, color=color).next_to(mob, direction, buff=buff)
            return l

        # ==========================================
        # 3. BUILD DIAGRAM
        # ==========================================
        # Coordinates
        Y_TOP, Y_GATES, Y_BOT = 2.0, 0.0, -2.5
        X_START, X_END = -4.5, 4.0
        X_F, X_I, X_C, X_O = -2.8, -1.2, 0.4, 2.0
        
        POS_MUL_F = [X_F, Y_TOP, 0]
        POS_MUL_I = [-0.4, 1.0, 0] 
        POS_ADD_C = [-0.4, Y_TOP, 0]
        POS_TANH  = [X_END-0.5, 0.5, 0]
        POS_MUL_O = [X_END-0.5, -1.5, 0]

        # Gates & Ops
        gate_f = create_gate_box(r"\sigma", "f", C_GATE, [X_F, Y_GATES, 0])
        gate_i = create_gate_box(r"\sigma", "i", C_GATE, [X_I, Y_GATES, 0])
        gate_c = create_gate_box(r"\tanh", r"\tilde{C}", C_TANH, [X_C, Y_GATES, 0])
        gate_o = create_gate_box(r"\sigma", "o", C_GATE, [X_O, Y_GATES, 0])
        
        op_mul_f = create_op(r"\times", POS_MUL_F)
        op_mul_i = create_op(r"\times", POS_MUL_I)
        op_add_c = create_op("+", POS_ADD_C)
        op_tanh  = create_gate_box("tanh", "", C_TANH, POS_TANH).scale(0.7)
        op_mul_o = create_op(r"\times", POS_MUL_O)

        # Wires & Labels
        wires = VGroup()
        lbls = VGroup()

        # --- Rails (Major labels increased to 42) ---
        w_c_in = Arrow([X_START, Y_TOP, 0], op_mul_f.get_left(), buff=0, color=C_LINE)
        lbls.add(add_label("C_{t-1}", w_c_in, UP, 0.15, 42, WHITE))
        
        w_c_mid1 = Line(op_mul_f.get_right(), op_add_c.get_left(), color=C_LINE)
        lbls.add(add_label(r"f_t \odot C_{t-1}", w_c_mid1, UP, 0.15))
        
        w_c_mid2 = Line(op_add_c.get_right(), [POS_TANH[0], Y_TOP, 0], color=C_LINE)
        lbls.add(add_label("C_t", w_c_mid2, UP, 0.15, 42, WHITE))
        
        w_c_out = Arrow([POS_TANH[0], Y_TOP, 0], [X_END+1, Y_TOP, 0], buff=0, color=C_LINE)
        lbls.add(add_label("Next C", w_c_out, UP, 0.15, 28, WHITE))

        w_h_rail = Line([X_START, Y_BOT, 0], [X_END+1, Y_BOT, 0], color=C_LINE)
        lbl_h = add_label("h_{t-1}, x_t", w_h_rail, UP, 0.15, 42, WHITE).move_to([X_START+1.5, Y_BOT+0.4, 0])
        lbls.add(lbl_h)

        # --- Feeds (Weights increased to 28) ---
        feeds = VGroup()
        for x_pos, name in zip([X_F, X_I, X_C, X_O], ["W_f", "W_i", "W_C", "W_o"]):
            l = Arrow([x_pos, Y_BOT, 0], [x_pos, Y_GATES - 0.4, 0], buff=0, color=C_LINE)
            d = Dot([x_pos, Y_BOT, 0], color=C_LINE, radius=0.06)
            t = MathTex(name, font_size=28, color=GRAY).next_to(l, LEFT, buff=0.08)
            feeds.add(VGroup(l, d, t))

        # --- Internal Connections ---
        w_f = Line(gate_f.get_top(), op_mul_f.get_bottom(), color=C_LINE)
        lbls.add(add_label("f_t", w_f, LEFT, 0.1))
        
        w_i = elbow_path(gate_i.get_top(), op_mul_i.get_left() + DOWN*0.1, direction="vertical_first")
        lbls.add(add_label("i_t", gate_i, UP+LEFT, 0.1))
        
        w_cand = elbow_path(gate_c.get_top(), op_mul_i.get_right() + DOWN*0.1, direction="vertical_first")
        lbls.add(add_label(r"\tilde{C}_t", gate_c, UP+RIGHT, 0.1, 32, C_TANH))
        
        w_update = Line(op_mul_i.get_top(), op_add_c.get_bottom(), color=C_LINE)
        lbls.add(add_label(r"i_t \odot \tilde{C}_t", w_update, RIGHT, 0.15))
        
        # --- Output Logic ---
        drop_start = [POS_TANH[0], Y_TOP, 0]
        d_drop = Dot(drop_start, radius=0.06, color=C_LINE)
        w_drop = Line(drop_start, op_tanh.get_top(), color=C_LINE)
        w_tanh_out = Line(op_tanh.get_bottom(), op_mul_o.get_top(), color=C_LINE)
        lbls.add(add_label(r"\tanh(C_t)", w_tanh_out, RIGHT, 0.15, 28, C_TANH))
        
        mid_x = (X_O + POS_TANH[0]) / 2 
        pts = [
            gate_o.get_top(),
            [mid_x, gate_o.get_top()[1], 0], 
            [mid_x, POS_MUL_O[1], 0],        
            op_mul_o.get_left()              
        ]
        w_o = VMobject().set_points_as_corners(pts).set_color(C_LINE).set_stroke(width=2)
        lbls.add(add_label("o_t", gate_o, UP, 0.15))
        
        w_h_out = Arrow(op_mul_o.get_right(), [X_END+1, POS_MUL_O[1], 0], buff=0, color=C_LINE)
        lbls.add(add_label("h_t", w_h_out, UP, 0.15, 42, WHITE))

        wires.add(w_c_in, w_c_mid1, w_c_mid2, w_c_out, w_h_rail, 
                  w_f, w_i, w_cand, w_update, 
                  w_drop, d_drop, w_tanh_out, w_o, w_h_out)

        diagram = VGroup(wires, feeds, gate_f, gate_i, gate_c, gate_o, 
                         op_mul_f, op_mul_i, op_add_c, op_tanh, op_mul_o, lbls)
        
        # ==========================================
        # 4. CODE & SCENE
        # ==========================================
        code_str = """def lstm_cell(x, h, c):
    combined = torch.cat((x, h), 1)

    # 1. Forget Gate
    # f_t = sigma(W_f @ [x, h])
    f_t = torch.sigmoid(combined @ W_f)

    # 2. Input Gate & Candidate
    # i_t = sigma(W_i @ [x, h])
    i_t = torch.sigmoid(combined @ W_i)
    # g_t = tanh(W_c @ [x, h])
    g_t = torch.tanh(combined @ W_c)

    # 3. Update Cell State
    c_new = f_t * c + i_t * g_t

    # 4. Output Gate
    # o_t = sigma(W_o @ [x, h])
    o_t = torch.sigmoid(combined @ W_o)
    
    # 5. Output Hidden State
    h_new = o_t * torch.tanh(c_new)

    return h_new, c_new"""

        filename = "lstm_temp.py"
        with open(filename, "w") as f:
            f.write(code_str)

        try:
            code_obj = Code(filename, language="python").scale(CODE_SCALE)
            code_obj.to_edge(LEFT, buff=0.5)

            # Title
            title = Text("LSTM Pytorch Implementation", font_size=42, weight=BOLD)
            title.to_edge(UP, buff=0.5)
            # Fix title to camera so it doesn't move when we zoom
            self.camera.frame.add(title) 

            diagram.move_to(DIAGRAM_SHIFT).scale(DIAGRAM_SCALE)
            self.play(Write(title), FadeIn(code_obj), FadeIn(diagram))
            self.wait(0.5)

            # --- CAMERA & HIGHLIGHTING ---
            def zoom_to_code(line_num, scale=1.1):
                if hasattr(code_obj, 'code'): lines = code_obj.code
                else: lines = code_obj[2]
                target = lines[line_num]
                self.play(
                    self.camera.frame.animate.move_to(target).set(width=config.frame_width/scale),
                    run_time=1.0
                )

            def zoom_to_diagram(mobject, scale=1.3):
                self.play(
                    self.camera.frame.animate.move_to(mobject).set(width=config.frame_width/scale),
                    run_time=1.0
                )

            def reset_view():
                self.play(
                    self.camera.frame.animate.move_to(ORIGIN).set(width=config.frame_width),
                    run_time=1.5
                )

            def get_highlighter(line_num):
                if hasattr(code_obj, 'code'): lines = code_obj.code
                else: lines = code_obj[2]
                return SurroundingRectangle(lines[line_num], color=C_HL, stroke_width=2, buff=0.05)

            # --- EXECUTION ---

            # 1. INPUT
            zoom_to_code(1, scale=1.5)
            hl = get_highlighter(1)
            self.play(Create(hl))
            
            zoom_to_diagram(wires[4], scale=1.2)
            self.play(Indicate(w_h_rail, color=C_HL))
            self.play(LaggedStart(*[Indicate(f, color=C_HL) for f in feeds], lag_ratio=0.1))

            # 2. FORGET GATE
            zoom_to_code(5, scale=1.5)
            self.play(Transform(hl, get_highlighter(5)))
            
            zoom_to_diagram(gate_f, scale=1.4)
            self.play(Indicate(gate_f, color=C_HL))
            animate_flow(w_f)
            self.play(Indicate(op_mul_f, scale_factor=1.3, color=RED))

            # 3. INPUT GATES
            zoom_to_code(9, scale=1.5)
            self.play(Transform(hl, get_highlighter(9)))
            
            zoom_to_diagram(gate_i, scale=1.4)
            self.play(Indicate(gate_i, color=C_HL))
            animate_flow(w_i)
            
            zoom_to_code(11, scale=1.5)
            self.play(Transform(hl, get_highlighter(11)))
            
            zoom_to_diagram(gate_c, scale=1.4)
            self.play(Indicate(gate_c, color=C_TANH))
            animate_flow(w_cand)

            # 4. UPDATE CELL STATE
            zoom_to_code(14, scale=1.5)
            self.play(Transform(hl, get_highlighter(14)))
            
            zoom_to_diagram(op_add_c, scale=1.4)
            self.play(Indicate(op_mul_i, color=RED))
            animate_flow(w_update)
            self.play(Indicate(op_add_c, scale_factor=1.3, color=RED))
            animate_flow(w_c_mid2)

            # 5. OUTPUT GATE
            zoom_to_code(18, scale=1.5)
            self.play(Transform(hl, get_highlighter(18)))
            
            zoom_to_diagram(gate_o, scale=1.4)
            self.play(Indicate(gate_o, color=C_HL))

            # 6. FINAL OUTPUT
            zoom_to_code(21, scale=1.5)
            self.play(Transform(hl, get_highlighter(21)))
            
            zoom_to_diagram(op_tanh, scale=1.3)
            animate_flow(w_drop)
            self.play(Indicate(op_tanh, color=C_TANH))
            animate_flow(w_tanh_out)
            
            path_trace = w_o.copy().set_stroke(color=C_HL, width=5, opacity=1)
            self.play(ShowPassingFlash(path_trace, time_width=0.5, run_time=1.0))
            
            self.play(Indicate(op_mul_o, scale_factor=1.3, color=RED))
            animate_flow(w_h_out)

            reset_view()
            self.wait(2)

        finally:
            if os.path.exists(filename):
                os.remove(filename)


%manim -qk -v warning LSTMDynamicFlow

Manim Community v0.19.0